# TP4 — Architecture Transformer
**Module :** IA Générative Multi Modale et Agentique  
**Étudiante :** Salma Riad

Travail basé sur le notebook `architecture_transformer.ipynb` du professeur : série sinusoïdale (RNN, LSTM, Transformer), démonstrations GPT et BERT, puis trois exercices notés. Exécuter les cellules dans l'ordre. **Les mesures et sorties ne sont pas préremplies : elles proviennent des exécutions réelles.**

In [ ]:
from pathlib import Path
import random, math, copy, zipfile
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# True : mode de validation plus rapide ; False : entraînements plus longs pour le rendu final.
MODE_RAPIDE = True
print('PyTorch:', torch.__version__, '| Appareil:', DEVICE, '| Mode rapide:', MODE_RAPIDE)

## Partie I — Prédire une sinusoïde : RNN, LSTM, Transformer

Entrée : **10 valeurs** successives de sin(x). Cible : la valeur suivante. Découpage chronologique, pas de mélange des exemples du test avec l'entraînement.

In [ ]:
serie = torch.sin(torch.linspace(0, 20, 200))
X_sin = torch.stack([serie[i:i+10] for i in range(len(serie)-10)]).unsqueeze(-1)
y_sin = torch.stack([serie[i+10] for i in range(len(serie)-10)]).unsqueeze(-1)
X_train_s, X_test_s = X_sin[:150], X_sin[150:]
y_train_s, y_test_s = y_sin[:150], y_sin[150:]
plt.figure(figsize=(9,2.5)); plt.plot(torch.linspace(0,20,200),serie); plt.title('Série sinusoïdale'); plt.show()
print('Entrées :', X_sin.shape, 'Cibles :', y_sin.shape)

In [ ]:
class SinRNN(nn.Module):
    def __init__(self):
        super().__init__(); self.seq=nn.RNN(1,16,batch_first=True); self.out=nn.Linear(16,1)
    def forward(self,x): return self.out(self.seq(x)[0][:,-1,:])

class SinLSTM(nn.Module):
    def __init__(self):
        super().__init__(); self.seq=nn.LSTM(1,16,batch_first=True); self.out=nn.Linear(16,1)
    def forward(self,x): return self.out(self.seq(x)[0][:,-1,:])

class SinTransformer(nn.Module):
    def __init__(self):
        super().__init__(); self.proj=nn.Linear(1,16); self.pos=nn.Embedding(10,16)
        layer=nn.TransformerEncoderLayer(16,2,dim_feedforward=32,dropout=0,batch_first=True)
        self.enc=nn.TransformerEncoder(layer,num_layers=1,enable_nested_tensor=False); self.out=nn.Linear(16,1)
    def forward(self,x):
        h=self.proj(x)+self.pos(torch.arange(x.size(1),device=x.device))[None,:,:]
        return self.out(self.enc(h)[:,-1,:])

def fit_sinus(modele, epochs):
    opt=torch.optim.Adam(modele.parameters(),lr=.01); loss_fn=nn.MSELoss()
    for epoch in range(epochs):
        modele.train(); opt.zero_grad(); loss=loss_fn(modele(X_train_s),y_train_s)
        loss.backward(); opt.step()
    modele.eval()
    with torch.no_grad():
        pred=modele(X_test_s).squeeze(-1); mse=loss_fn(pred,y_test_s.squeeze(-1)).item()
    return pred.numpy(),mse

resultats_sinus={}
for nom,classe in [('RNN',SinRNN),('LSTM',SinLSTM),('Transformer',SinTransformer)]:
    pred,mse=fit_sinus(classe(),epochs=35 if MODE_RAPIDE else 300)
    resultats_sinus[nom]=mse
    plt.figure(figsize=(8,2.2)); plt.plot(y_test_s.numpy(),label='Valeur réelle');plt.plot(pred,label=nom)
    plt.title(f'{nom} — MSE test : {mse:.5f}');plt.legend();plt.show()
print('Comparaison MSE test :',resultats_sinus)

## Partie I (suite) — Exemples GPT et BERT préentraînés

Le support illustre GPT-2 pour prolonger une phrase et BERT multilingue pour prédire un token `[MASK]`. Ces exemples **optionnels** nécessitent `transformers` et un téléchargement initial des modèles. Ils sont désactivés par défaut pour éviter d'imposer plusieurs Go de téléchargements.

In [ ]:
EXECUTER_MODELES_PREENTRAINES = False
if EXECUTER_MODELES_PREENTRAINES:
    from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM
    tok=AutoTokenizer.from_pretrained('gpt2')
    model=AutoModelForCausalLM.from_pretrained('gpt2').eval()
    inputs=tok('Artificial intelligence can help people',return_tensors='pt')
    with torch.no_grad():
        result=model.generate(**inputs,max_new_tokens=35,do_sample=True,temperature=.8,pad_token_id=tok.eos_token_id)
    print('GPT-2 :',tok.decode(result[0],skip_special_tokens=True))
    del model
    tok=AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
    model=AutoModelForMaskedLM.from_pretrained('bert-base-multilingual-cased').eval()
    phrase=f'Rabat est la capitale du {tok.mask_token}.'
    inputs=tok(phrase,return_tensors='pt')
    with torch.no_grad(): scores=model(**inputs).logits
    mask_idx=(inputs['input_ids'][0]==tok.mask_token_id).nonzero(as_tuple=True)[0].item()
    top=torch.topk(scores[0,mask_idx],5).indices.tolist()
    print('BERT — tokens prédits :', [tok.decode([x]).strip() for x in top])
else:
    print('Exemples préentraînés facultatifs non exécutés ; définir EXECUTER_MODELES_PREENTRAINES=True pour les lancer.')

---
## Exercice 1 — Consommation électrique : EnergyTransformer

Source : *Individual Household Electric Power Consumption* (UCI). Variable `Global_active_power` en kW, agrégée en moyenne horaire ; au moins 45 minutes renseignées. Entrée de 24 heures, prédiction de l'heure suivante. Séparation chronologique **70 % / 15 % / 15 %**, normalisation **sur l'entraînement seulement**, suppression des fenêtres incomplètes. L'archive fournie est incluse dans le dossier `data/`.

In [ ]:
data_dir=Path('data')
archive=data_dir/'household_power_consumption.zip'
raw=data_dir/'household_power_consumption.txt'
if not raw.exists():
    if not archive.exists():
        raise FileNotFoundError(f'Dataset absent : placer household_power_consumption.zip dans {data_dir.resolve()}')
    with zipfile.ZipFile(archive) as z:
        names=[n for n in z.namelist() if n.endswith('household_power_consumption.txt')]
        if not names: raise ValueError('Le ZIP ne contient pas household_power_consumption.txt')
        with z.open(names[0]) as src, raw.open('wb') as dst:
            import shutil
            shutil.copyfileobj(src,dst)
print('Dataset disponible :',raw, '| taille (Mo) :',round(raw.stat().st_size/1e6,1))

In [ ]:
df=pd.read_csv(raw,sep=';',usecols=['Date','Time','Global_active_power'],na_values=['?'],low_memory=False)
dates=pd.to_datetime(df['Date']+' '+df['Time'],format='%d/%m/%Y %H:%M:%S')
puissance=pd.Series(pd.to_numeric(df['Global_active_power'],errors='coerce').to_numpy(),index=pd.DatetimeIndex(dates))
del df
horaire=puissance.resample('h').agg(['mean','count'])
serie_h=horaire['mean'].where(horaire['count']>=45)
n=len(serie_h); fin_train=int(.70*n); fin_val=int(.85*n)
moyenne=float(serie_h.iloc[:fin_train].mean()); ecart_type=float(serie_h.iloc[:fin_train].std())
if not np.isfinite(ecart_type) or ecart_type<=0: raise ValueError('Normalisation impossible')
valeurs=((serie_h-moyenne)/ecart_type).to_numpy(dtype=np.float32)
X_list=[]; y_list=[]; dates_cibles=[]
for i in range(24,n):
    seq=valeurs[i-24:i]; cible=valeurs[i]
    if np.isfinite(seq).all() and np.isfinite(cible):
        X_list.append(seq); y_list.append(cible); dates_cibles.append(i)
X_energy=torch.from_numpy(np.asarray(X_list,dtype=np.float32)).unsqueeze(-1)
y_energy=torch.tensor(y_list,dtype=torch.float32).unsqueeze(-1)
dates_cibles=np.asarray(dates_cibles)
mask_train=dates_cibles<fin_train
mask_val=(dates_cibles>=fin_train)&(dates_cibles<fin_val)
mask_test=dates_cibles>=fin_val
assert mask_train.sum()>0 and mask_val.sum()>0 and mask_test.sum()>0
train_loader=DataLoader(TensorDataset(X_energy[mask_train],y_energy[mask_train]),batch_size=64,shuffle=True)
val_loader=DataLoader(TensorDataset(X_energy[mask_val],y_energy[mask_val]),batch_size=64)
test_loader=DataLoader(TensorDataset(X_energy[mask_test],y_energy[mask_test]),batch_size=64)
print('Fenêtres valides :',len(X_energy),'| Train/Val/Test :',mask_train.sum(),mask_val.sum(),mask_test.sum())
print('Statistiques entraînement :',moyenne,ecart_type)

In [ ]:
class EnergyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection=nn.Linear(1,32)
        self.positions=nn.Embedding(24,32)
        layer=nn.TransformerEncoderLayer(d_model=32,nhead=4,dim_feedforward=128,dropout=.1,batch_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=2,enable_nested_tensor=False)
        self.head=nn.Linear(32,1)
    def forward(self,x):
        if x.ndim!=3 or x.size(1)!=24 or x.size(2)!=1: raise ValueError('Forme attendue : (batch,24,1)')
        h=self.projection(x)+self.positions(torch.arange(24,device=x.device)).unsqueeze(0)
        return self.head(self.encoder(h)[:,-1,:])

energy=EnergyTransformer().to(DEVICE)
with torch.no_grad():
    assert energy(torch.zeros(2,24,1,device=DEVICE)).shape==(2,1)
print('EnergyTransformer : projection 1→32, 2 couches, 4 têtes, sortie (batch,1)')

In [ ]:
def eval_energy(loader):
    energy.eval(); total=0.; n_examples=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE);y=y.to(DEVICE)
            total+=nn.functional.mse_loss(energy(x),y,reduction='sum').item();n_examples+=len(x)
    return total/n_examples

opt_energy=torch.optim.Adam(energy.parameters(),lr=.001)
epochs_energy=3 if MODE_RAPIDE else 20
best_val=float('inf'); best_state=None; history=[]
for epoch in range(epochs_energy):
    energy.train(); somme=0.; total=0
    for xx,yy in train_loader:
        xx=xx.to(DEVICE); yy=yy.to(DEVICE)
        opt_energy.zero_grad();pred=energy(xx); loss=nn.functional.mse_loss(pred,yy)
        loss.backward();opt_energy.step();somme+=loss.item()*len(xx);total+=len(xx)
    val=eval_energy(val_loader);train_loss=somme/total;history.append((train_loss,val))
    if val<best_val:
        best_val=val; best_state=copy.deepcopy(energy.state_dict())
    print(f'Époque {epoch+1}/{epochs_energy} | MSE train={train_loss:.5f} | MSE val={val:.5f}')
energy.load_state_dict(best_state)
Path('models').mkdir(exist_ok=True)
torch.save(best_state,'models/energy_transformer_best.pth')
print('Meilleure MSE validation :',best_val)

In [ ]:
energy.eval();pred_norm=[]
with torch.no_grad():
    for xx,_ in test_loader: pred_norm.extend(energy(xx.to(DEVICE)).cpu().numpy().reshape(-1))
pred_norm=np.asarray(pred_norm)
y_true=(y_energy[mask_test].numpy().reshape(-1)*ecart_type+moyenne)
y_pred=pred_norm*ecart_type+moyenne
X_test_np=X_energy[mask_test].numpy().squeeze(-1)
baseline_last=X_test_np[:,-1]*ecart_type+moyenne
baseline_day=X_test_np[:,0]*ecart_type+moyenne

def metrics(true,pred):
    return float(np.mean(np.abs(true-pred))),float(np.sqrt(np.mean((true-pred)**2)))
score_energy=pd.DataFrame({name:dict(zip(['MAE (kW)','RMSE (kW)'],metrics(y_true,pred)))
    for name,pred in [('Transformer',y_pred),('Persistance',baseline_last),('Référence J-1',baseline_day)]}).T
print(score_energy.round(4).to_string())
plt.figure(figsize=(12,3));length=min(168,len(y_true))
plt.plot(y_true[:length],label='Observation');plt.plot(y_pred[:length],label='Transformer',alpha=.8)
plt.plot(baseline_last[:length],label='Persistance',alpha=.6)
plt.title('Test : 7 jours consécutifs disponibles');plt.xlabel('Heures');plt.ylabel('kW');plt.legend();plt.show()

### Réponses — Exercice 1

- **Amélioration par rapport aux références :** interpréter les valeurs MAE et RMSE réellement affichées ci-dessus ; le Transformer n'est pas automatiquement meilleur.
- **Fenêtre de 168 h :** pourrait capturer les cycles hebdomadaires, mais augmente le coût d'attention (quadratique en longueur).
- **Heure et jour :** des variables calendaires aideraient à distinguer pics journaliers et comportement de semaine/week-end.
- **Normalisation train uniquement :** éviter une fuite d'informations du futur (validation/test).
- **kWh plutôt que kW :** calculer pour chaque heure l'énergie intégrée en kWh ; avec une puissance moyenne horaire en kW sur 1 h, valeur numérique approximativement égale à kW × 1 h.

---
## Exercice 2 — Mini-GPT autorégressif par caractères

Embeddings de dimension 64, deux blocs, quatre têtes, LayerNorm avant l'attention, MLP 64→256→64 avec GELU, connexions résiduelles et masque causal. Prédiction de chaque caractère suivant. Génération de **200 caractères** aux températures 0.5, 1.0 et 1.5.

In [ ]:
corpus_path=Path('corpus.txt')
texte=corpus_path.read_text(encoding='utf-8')
caracteres=sorted(set(texte));stoi={c:i for i,c in enumerate(caracteres)};itos={i:c for c,i in stoi.items()}
tokens=torch.tensor([stoi[c] for c in texte],dtype=torch.long)
SEQ_LEN=64; VOCAB=len(stoi)
assert len(tokens)>SEQ_LEN+2
print('Corpus :',len(texte),'caractères, vocabulaire de',VOCAB,'symboles')

In [ ]:
class BlocGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1=nn.LayerNorm(64)
        self.attn=nn.MultiheadAttention(embed_dim=64,num_heads=4,batch_first=True,dropout=0)
        self.ln2=nn.LayerNorm(64)
        self.mlp=nn.Sequential(nn.Linear(64,256),nn.GELU(),nn.Linear(256,64))
    def forward(self,x):
        T=x.size(1)
        mask=torch.triu(torch.ones(T,T,dtype=torch.bool,device=x.device),diagonal=1)
        h=self.ln1(x)
        a,_=self.attn(h,h,h,attn_mask=mask,need_weights=False)
        x=x+a
        return x+self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self,vocab):
        super().__init__()
        self.tok=nn.Embedding(vocab,64);self.pos=nn.Embedding(SEQ_LEN,64)
        self.blocks=nn.Sequential(BlocGPT(),BlocGPT())
        self.norm=nn.LayerNorm(64); self.head=nn.Linear(64,vocab)
    def forward(self,ids):
        if ids.size(1)>SEQ_LEN: raise ValueError('Longueur maximale : 64')
        pos=torch.arange(ids.size(1),device=ids.device)
        return self.head(self.norm(self.blocks(self.tok(ids)+self.pos(pos)[None,:,:])))

gpt=MiniGPT(VOCAB).to(DEVICE)
with torch.no_grad():assert gpt(torch.zeros(2,64,dtype=torch.long,device=DEVICE)).shape==(2,64,VOCAB)
print('Formes GPT vérifiées')

In [ ]:
# Test de causalité : changer des tokens futurs ne doit pas changer les logits passés.
gpt.eval()
base=tokens[:64].unsqueeze(0).to(DEVICE).clone();modifie=base.clone();modifie[:,32:]=(modifie[:,32:]+1)%VOCAB
with torch.no_grad():
    diff=(gpt(base)[:,:32,:]-gpt(modifie)[:,:32,:]).abs().max().item()
print('Différence maximale avant position 32 :',diff)
assert diff<1e-5, 'Fuite des informations futures dans le masque causal'

In [ ]:
opt_gpt=torch.optim.AdamW(gpt.parameters(),lr=.002)
iterations=80 if MODE_RAPIDE else 500
batch=16
for step in range(iterations):
    gpt.train()
    debut=torch.randint(0,len(tokens)-SEQ_LEN-1,(batch,))
    x=torch.stack([tokens[k:k+SEQ_LEN] for k in debut]).to(DEVICE)
    y=torch.stack([tokens[k+1:k+SEQ_LEN+1] for k in debut]).to(DEVICE)
    opt_gpt.zero_grad(); logits=gpt(x)
    loss=nn.functional.cross_entropy(logits.reshape(-1,VOCAB),y.reshape(-1))
    loss.backward();opt_gpt.step()
    if (step+1)%max(1,iterations//4)==0: print('Itération',step+1,'| perte :',round(loss.item(),4))
Path('models').mkdir(exist_ok=True)
torch.save(gpt.state_dict(),'models/mini_gpt.pth')

In [ ]:
@torch.no_grad()
def generer_gpt(amorce='La ',n=200,temperature=1.):
    gpt.eval(); ids=[stoi.get(c,stoi.get(' ',0)) for c in amorce]
    for _ in range(n):
        contexte=torch.tensor([ids[-SEQ_LEN:]],dtype=torch.long,device=DEVICE)
        logits=gpt(contexte)[0,-1]/temperature
        choix=torch.multinomial(torch.softmax(logits,dim=-1),1).item()
        ids.append(choix)
    return ''.join(itos[j] for j in ids)
for temp in [.5,1.,1.5]:
    print('Température',temp,':', generer_gpt(n=200,temperature=temp))

### Réponses — Exercice 2

- **Sans masque causal :** le modèle voit les caractères futurs pendant l'entraînement, ce qui fausse la prédiction autorégressive.
- **Température :** 0,5 rend les sorties plus concentrées/prévisibles ; 1,5 augmente la diversité et souvent les erreurs ; 1,0 conserve l'échelle des logits.
- **Cibles décalées :** pour que chaque position apprenne à deviner la suivante sans recopier son entrée.

---
## Exercice 3 — Mini-BERT et prédiction de mots masqués

Tokenisation par mots, séquences de 32, vocabulaire construit sur le train. Tokens spéciaux `PAD=0`, `UNK=1`, `CLS=2`, `SEP=3`, `MASK=4`. Sélection de 15 % des tokens ordinaires : 80 % `[MASK]`, 10 % mot aléatoire, 10 % inchangé. **Perte uniquement sur les positions choisies**, avec `ignore_index=-100`. Attention bidirectionnelle et masque de padding.

In [ ]:
phrases=[l.strip().lower() for l in Path('corpus_bert.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
random.Random(SEED).shuffle(phrases)
part=max(1,int(.8*len(phrases))); phrases_train=phrases[:part];phrases_val=phrases[part:]
PAD,UNK,CLS,SEP,MASK=0,1,2,3,4
special=['[PAD]','[UNK]','[CLS]','[SEP]','[MASK]']
mots=sorted(set(m for phrase in phrases_train for m in phrase.split()))
vocab={t:i for i,t in enumerate(special+mots)};inv_vocab={i:t for t,i in vocab.items()}
MAX_LEN=32

def encoder_bert(phrase):
    ids=[CLS]+[vocab.get(m,UNK) for m in phrase.split()][:MAX_LEN-2]+[SEP]
    return ids+[PAD]*(MAX_LEN-len(ids))
train_ids=torch.tensor([encoder_bert(p) for p in phrases_train],dtype=torch.long)
val_ids=torch.tensor([encoder_bert(p) for p in phrases_val],dtype=torch.long)
print('Phrases train/val :',len(train_ids),len(val_ids),'| vocabulaire :',len(vocab))

In [ ]:
def masquer_bert(input_ids,generator=None):
    corrompu=input_ids.clone();labels=torch.full_like(input_ids,-100)
    for row in range(len(input_ids)):
        candidats=((input_ids[row]!=PAD)&(input_ids[row]!=CLS)&(input_ids[row]!=SEP)&(input_ids[row]!=MASK)).nonzero(as_tuple=True)[0]
        if not len(candidats): continue
        n_select=max(1,round(.15*len(candidats)))
        selection=candidats[torch.randperm(len(candidats),generator=generator)[:n_select]]
        labels[row,selection]=input_ids[row,selection]
        for col in selection.tolist():
            p=torch.rand((),generator=generator).item()
            if p<.8:corrompu[row,col]=MASK
            elif p<.9:corrompu[row,col]=int(torch.randint(5,len(vocab),(1,),generator=generator).item()) if len(vocab)>5 else UNK
            # Sinon : garder le token original.
    return corrompu,labels

assert (masquer_bert(train_ids[:1])[1]!=-100).sum()>0

In [ ]:
class MiniBERT(nn.Module):
    def __init__(self,vocab_size):
        super().__init__();self.emb=nn.Embedding(vocab_size,64,padding_idx=PAD);self.pos=nn.Embedding(MAX_LEN,64)
        layer=nn.TransformerEncoderLayer(d_model=64,nhead=4,dim_feedforward=256,dropout=0.0,batch_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=2,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(64);self.head=nn.Linear(64,vocab_size)
    def forward(self,ids,padding_mask=None):
        if padding_mask is None: padding_mask=ids.eq(PAD)
        pos=torch.arange(ids.size(1),device=ids.device)
        h=self.emb(ids)+self.pos(pos)[None,:,:]
        h=self.encoder(h,src_key_padding_mask=padding_mask)
        return self.head(self.norm(h))
bert=MiniBERT(len(vocab)).to(DEVICE)
with torch.no_grad():assert bert(train_ids[:2].to(DEVICE)).shape==(2,MAX_LEN,len(vocab))

In [ ]:
# Test : modifier les valeurs des positions PAD sans changer le masque de padding
bert.eval();base=val_ids[:1].to(DEVICE).clone()
mask=base.eq(PAD)
changed=base.clone();changed[mask]=UNK
with torch.no_grad():
    res1=bert(base,padding_mask=mask);res2=bert(changed,padding_mask=mask)
    diff=(res1[~mask]-res2[~mask]).abs().max().item()
print('Différence sur tokens non-PAD :',diff)
assert diff<1e-5, 'Le padding influence les positions utiles'

In [ ]:
opt_bert=torch.optim.AdamW(bert.parameters(),lr=.002)
epochs_bert=10 if MODE_RAPIDE else 50
batch_size=16
for epoch in range(epochs_bert):
    bert.train();perm=torch.randperm(len(train_ids));losses=[]
    for indices in perm.split(batch_size):
        x,labels=masquer_bert(train_ids[indices]);x=x.to(DEVICE);labels=labels.to(DEVICE)
        opt_bert.zero_grad();logits=bert(x)
        loss=nn.functional.cross_entropy(logits.reshape(-1,len(vocab)),labels.reshape(-1),ignore_index=-100)
        loss.backward();opt_bert.step();losses.append(loss.item())
    if (epoch+1)%max(1,epochs_bert//5)==0:print('Epoch',epoch+1,'| perte train',round(float(np.mean(losses)),4))
Path('models').mkdir(exist_ok=True)
torch.save(bert.state_dict(),'models/mini_bert.pth')

In [ ]:
bert.eval();gen=torch.Generator().manual_seed(123)
val_input,val_labels=masquer_bert(val_ids,generator=gen)
with torch.no_grad():
    logits=bert(val_input.to(DEVICE)).cpu()
positions=val_labels!=-100
exactitude=(logits.argmax(dim=-1)[positions]==val_labels[positions]).float().mean().item()
print(f'Accuracy validation sur tokens sélectionnés : {exactitude:.2%} ({int(positions.sum())} tokens)')
for i,j in positions.nonzero()[:5].tolist():
    top=torch.topk(logits[i,j],k=min(3,len(vocab))).indices.tolist()
    print('Phrase :',phrases_val[i],'| Réel :',inv_vocab[int(val_labels[i,j])], '| Top-3 :',[inv_vocab[k] for k in top])

### Réponses — Exercice 3

- **Contexte à droite :** BERT utilise une attention bidirectionnelle, contrairement au masque causal GPT.
- **Perte sur positions sélectionnées :** entraîner la reconstruction des mots ciblés, pas une copie triviale de tous les tokens visibles.
- **Pas une génération GPT :** l'objectif est de retrouver des tokens masqués à partir des deux côtés, sans procédure autorégressive de prédiction du prochain token.

---
## Bilan

Les courbes, tableaux, exactitudes et générations ci-dessus sont calculés **à l'exécution**. Le mode rapide sert à vérifier le fonctionnement ; pour des expérimentations plus longues, passer `MODE_RAPIDE=False`, puis **Restart & Run All**. Les modèles entraînés sont enregistrés dans `models/` (ignorés par Git).